In [0]:
dbutils.widgets.text("catalog","fk_1361")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
select * from fk_1361.silver.products_cleaned

In [0]:
spark.sql(f"""
create table if not exists {catalog}.silver.products_scd as
select product_id, product_name, brand, category.l1 as category_l1, category.l2 as category_l2, pack.size as pack_size, pack.unit as pack_unit, mrp, selling_price, is_active, tags, snapshot_date as last_changed_snapshot_date
from {catalog}.silver.products_cleaned
where 1 = 0
""")

In [0]:
spark.sql(f"""
merge into {catalog}.silver.products_scd t
using (
    select product_id, product_name, brand, category.l1 as category_l1, category.l2 as category_l2, pack.size as pack_size, pack.unit as pack_unit, mrp, selling_price, is_active, tags, snapshot_date as last_changed_snapshot_date from {catalog}.silver.products_cleaned
    qualify row_number() over (partition by product_id order by snapshot_date desc) = 1
) s
on t.product_id = s.product_id
when matched then update set *
when not matched then 
insert (product_id, product_name, brand, category_l1, category_l2, pack_size, pack_unit, mrp, selling_price, is_active, tags, last_changed_snapshot_date) values (s.product_id, s.product_name, s.brand, s.category_l1, s.category_l2, s.pack_size, s.pack_unit, s.mrp, s.selling_price, s.is_active, s.tags, s.last_changed_snapshot_date)
""")